# 16.5 n960 - Occlusal alignment comparison

Exploratory source-view experiment on the frozen 16.3 dataset. Source crowding includes posterior regions. Results do not measure frontal phone-photo performance. Validation contains 38 target-positive images and six diastema instances.

## Install dependencies

Pin the training library for reproducibility.

In [ ]:
%pip install -q ultralytics==8.3.228

## Stage and verify source data

Kaggle expands ZIP uploads. Verify each staged image and label using the frozen manifest.

In [ ]:
from pathlib import Path
import json,hashlib,shutil,time
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image,ImageDraw
from ultralytics import YOLO
candidates=list(Path('/kaggle/input').rglob('manifest.json'))
matches=[]
for candidate in candidates:
    try:
        content=json.loads(candidate.read_text())
        if isinstance(content,list) and len(content)==662 and content[0].get('image_sha256'): matches.append(candidate)
    except (ValueError,AttributeError):
        pass
assert len(matches)==1,matches
source=matches[0].parent
manifest=json.loads(matches[0].read_text())
target=Path('/tmp/alignment_occlusal')
shutil.copytree(source,target,dirs_exist_ok=True)
for row in manifest:
    name=f"{row['split']}_{row['id']}"
    for kind,ext,field in [('images','.jpg','image_sha256'),('labels','.txt','label_sha256')]:
        path=target/kind/row['split']/(name+ext)
        assert hashlib.sha256(path.read_bytes()).hexdigest()==row[field]
print('Source ZIP SHA256 (local publication):', '3e77ff9d512a55cd8cde3951122478964b818fc077fe951ab390b1511590cab4')
display(pd.DataFrame(manifest).groupby('split').agg(images=('id','size'),boxes=('boxes','sum')))
(target/'data.yaml').write_text(f'path: {target}\ntrain: images/train\nval: images/val\nnames:\n  0: midline_diastema\n  1: source_crowding\n')
assert torch.cuda.is_available()
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Inspect source annotations

Yellow is midline diastema. Cyan is source crowding.

In [ ]:
path=sorted((target/'images/train').glob('*.jpg'))[0]
image=Image.open(path).convert('RGB');draw=ImageDraw.Draw(image)
for line in (target/'labels/train'/(path.stem+'.txt')).read_text().splitlines():
    cls,x,y,w,h=map(float,line.split())
    draw.rectangle(((x-w/2)*image.width,(y-h/2)*image.height,(x+w/2)*image.width,(y+h/2)*image.height),outline=['yellow','cyan'][int(cls)],width=3)
plt.figure(figsize=(8,6));plt.imshow(image);plt.axis('off');plt.show()

## Train YOLO11n

Fixed 60 epochs with a five-hour epoch-boundary limit and periodic checkpoints. This arm uses yolo11n.pt, 960 pixels and batch 8.

In [ ]:
model=YOLO('yolo11n.pt')
started=time.monotonic()
def stop_after_five_hours(trainer):
    if time.monotonic()-started>5*3600: trainer.stop=True
model.add_callback('on_train_epoch_end',stop_after_five_hours)
result=model.train(data=str(target/'data.yaml'),epochs=60,patience=0,imgsz=960,batch=8,device=0,workers=2,seed=42,deterministic=True,save=True,save_period=10,plots=True,project='/kaggle/working/runs',name='occlusal_n960')
print(result.results_dict)

## Inspect validation and checkpoint

Report source-view metrics only. Validation contains 38 target-positive images and may contain related scenes.

In [ ]:
run=Path(result.save_dir)
history=pd.read_csv(run/'results.csv');history.columns=history.columns.str.strip()
display(history.tail())
history.plot(x='epoch',y=['metrics/mAP50(B)','metrics/mAP50-95(B)'],grid=True);plt.show()
print('Best checkpoint SHA256:',hashlib.sha256((run/'weights/best.pt').read_bytes()).hexdigest())
(run/'metrics.json').write_text(json.dumps(result.results_dict,indent=2))